The goal of this notebook is to analyze if there is any relationship between these 3 columns:
- `@/@@_class`: SMILES parity tag (SMILES formatting convention)
- `R/S_class`: CIP absolute configuration (true chemical property)
- `F/L_class`: Chromatographic elution order (chemical conseuence of chirality)

Performs pairwise association between the 3 columns to determine if they are genuinely independent or if there is a relationship.

Focus on 
1. Chi-square test + p-value    (hypothesis test: are variables independent?)
2. Cramér's V                   (effect size of chi-square; no label mapping needed)
3. Cohen's kappa (both mappings) (chance-corrected agreement; mapping-sensitive)
4. MCC / phi coefficient        (signed correlation; mapping-sensitive)
5. Pearson correlation          (equivalent to MCC for binary data; mapping-sensitive)

Spearman rank correlation would NOT be relevant here because it requires ordinal data, meaning the categories have a meaningful order. Here, the 3 columns are binary variables so there is no natural ordering. 


**References:**
- Cohen (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement.*
- Cramér (1946). *Mathematical Methods of Statistics.* Princeton University Press.
- Chicco & Jurman (2020). The advantages of MCC over F1. *BMC Genomics.* https://doi.org/10.1186/s12864-019-6413-7
- `scipy.stats.chi2_contingency`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html
- `sklearn.metrics.matthews_corrcoef`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.matthews_corrcoef.html

In [1]:
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pearsonr
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef

In [2]:
# load stereoisomer data
# https://github.com/jairesdesousa/chiraldlsv/blob/main/TSNE_maps/class_all.csv
input_path = "class_all.csv"
df = pd.read_csv(input_path, index_col=0)
df

,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R
...,...,...,...,...,...,...
3853,OC[C@]1(CCCOCc2ccccc2)COCCO1,OC[C@@]1(CCCOCc2ccccc2)COCCO1,TR,L,@,R
3854,OCCCC[C@H](O)c1ccc(F)cc1,OCCCC[C@@H](O)c1ccc(F)cc1,TR,F,@,S
3855,OCCCC[C@@H](O)c1ccc(F)cc1,OCCCC[C@H](O)c1ccc(F)cc1,TR,L,@@,R
3856,OCCCC[C@]1(CO)COCCO1,OCCCC[C@@]1(CO)COCCO1,TR,L,@,S


# Metric Explanations

### Metrics that require NO label mapping

**Chi-square test + p-value**
Tests the null hypothesis that two categorical variables are statistically independent.
The chi-square statistic measures how far the observed cell counts in the contingency table deviate from what we'd expect if the variables were completely unrelated.
- Range: chi-square in [0, ∞); p-value in [0, 1]
- Interpretation: A small p-value (typically < 0.05) rejects independence i.e., the variables are associated. However, chi-square alone is sensitive to sample size: with n=3858, even a trivially weak association will be statistically significant. **Always pair with Cramér's V for effect size.**
- **No label mapping needed.** Uses raw category counts directly.
- Reference: `scipy.stats.chi2_contingency` https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html

**Cramér's V**
Measures the strength of association between two nominal variables, normalized from the chi-square statistic.
The effect size of the chi-square test. 
Answers: *how strongly are these variables associated, regardless of how the categories are labelled?*
- Formula: `V = sqrt(chi2 / (n * min(rows-1, cols-1)))`
- Range: [0, 1]. 0 = no association; 1 = perfect association.
- Interpretation: Cohen 1988 guidelines for 2x2 tables gives this guidelines: ~0.10 = small, ~0.30 = medium, ~0.50 = large
- **V Cannot be negative** — measures strength only, not direction.
- **No label mapping needed.** This is its key advantage.
References:
- Cramér (1946). Mathematical Methods of Statistics.
- Wikipedia: https://en.wikipedia.org/wiki/Cram%C3%A9r%27s_V

### Metrics that require a label mapping (encoding assumption)

**Cohen's kappa**
Measures agreement between two categorical labellings of the same items, corrected for the agreement expected by chance alone. Inter-annotator agreement.
- If @/@@ and R/S are truly associated, then the two mappings should produce a positive and negative kappa respectively of equal magnitude.
- If instead, both produce kappa ≈ 0, there is strong evidence that there is no meaningful relationship under either assumption, and the result is robust to the mapping choice.
- kappa still requires you to treat the two columns as if they are measuring the same underlying thing on the same items — i.e., inter-annotator agreement. **Cramér's V remains preferable as the primary measure because it sidesteps the mapping question entirely.**


Chance-corrected agreement between two labellings of the same items.
- Formula: `κ = (p_observed - p_chance) / (1 - p_chance)`
- Range: [-1, 1]. 0 = chance-level agreement; 1 = perfect agreement; negative = systematic disagreement beyond chance.
- Interpretation from Landis & Koch (1977):
    - <0 poor
    - 0–0.20 slight
    - 0.21–0.40 fair
    - 0.41–0.60 moderate
    - 0.61–0.80 substantial
    - 0.81–1.0 almost perfect
- **Label mapping is needed**: Kappa requires both columns to use the same category names, or an explicit encoding that maps them to shared integers. The mapping is a scientific assumption — changing it negates the sign of kappa. We therefore report kappa under BOTH possible mappings as a sensitivity check.
- ⚠️ Sign depends on label encoding. We report under both mappings as a sensitivity check.
- For balanced (50/50) binary variables: **kappa = MCC** exactly. Kappa and MCC diverge only under class imbalance.
References:
- Cohen (1960). Educational and Psychological Measurement.
- sklearn: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html

**MCC (Matthews Correlation Coefficient aka phi coefficient)**
Measures the Pearson correlation between two binary variables. Signed — captures direction.
For binary classifications, the phi coefficient equals the Matthews Correlation Coefficient.
- Formula: `MCC = (TP·TN - FP·FN) / sqrt((TP+FP)(TP+FN)(TN+FP)(TN+FN))`
- Range: [-1, 1]. 0 = no association; ±1 = perfect association (positive or inverse) i.e., one variable perfectly predicts the other.
- For balanced data: MCC = kappa. **For imbalanced data: MCC is more reliable (Chicco & Jurman 2020).**
- ⚠️ Sign depends on label encoding. Report under both mappings.
    - It is a signed measure of association — it captures both the strength and the direction of the relationship.
- **Label mapping is needed**: Flipping the encoding of one variable negates MCC.
References:
- Matthews (1975). Biochimica et Biophysica Acta.
- sklearn: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.matthews_corrcoef.html
- Chicco & Jurman (2020): https://doi.org/10.1186/s12864-019-6413-7


**Pearson r**
Measures the linear correlation between variables. For binary variables, Pearson r is mathematically identical to MCC (phi coefficient).
While Pearson is typially used for continuous variables, it is included here for completeness and to confirm that MCC and Pearson agree on binary data.
- Range: [-1, 1]. 0 = no association; ±1 = perfect association (positive or inverse) i.e., one variable perfectly predicts the other.
- **Label mapping is needed**
Reference:
- `scipy.stats.pearsonr` https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html

# First print pairwise contigency table
This counts how often each combination of values co-occurs across two categorical variables.
It is the foundational data structure for all metrics below.

Expected count per cell under independence = `(row_total × col_total) / n`.

Results show approximately even distribution for each pair of variales. 
- The chi-square test will measure whether these counts deviate significantly from what we'd expect if the two variables were completely independent
- Cramér's V normalizes the chi-square statistic by sample size to give an effect size on [0, 1].
- Cohen's kappa and MCC are computed from the same 2×2 table but interpreted differently.

In [3]:
# pairwise cross-tabulations
print("=" * 60)
print("Pairwise cross-tabulations (raw counts)")
print("=" * 60)
pairs = [
    ("@/@@_class", "R/S_class"),
    ("@/@@_class", "F/L_class"),
    ("R/S_class",  "F/L_class"),
]
for col_a, col_b in pairs:
    print('*'*33)
    print(f"{col_a}  vs  {col_b}:")
    print(pd.crosstab(df[col_a], df[col_b]))
    print()

Pairwise cross-tabulations (raw counts)
*********************************
@/@@_class  vs  R/S_class:
R/S_class     R    S
@/@@_class          
@           983  946
@@          946  983

*********************************
@/@@_class  vs  F/L_class:
F/L_class      F     L
@/@@_class            
@           1018   911
@@           911  1018

*********************************
R/S_class  vs  F/L_class:
F/L_class     F     L
R/S_class            
R          1016   913
S           913  1016



# Metric functions

In [4]:
def cramers_v(col_a: pd.Series, col_b: pd.Series) -> tuple[float, float]:
    """
    Compute Cramér's V and chi-square p-value for two nominal columns.

    Args:
        col_a: First categorical column (raw string labels).
        col_b: Second categorical column (raw string labels).

    Returns:
        Tuple of (v, p_value) where v is Cramér's V in [0, 1] and
        p_value is the chi-square test p-value.
    """
    table = pd.crosstab(col_a, col_b).values
    chi2, p, _, _ = chi2_contingency(table)
    n = table.sum()
    min_dim = min(table.shape) - 1
    return float(np.sqrt(chi2 / (n * min_dim))), float(p)

In [5]:
def chi_square(col_a: pd.Series, col_b: pd.Series) -> tuple[float, float]:
    """
    Compute chi-square statistic and p-value for two nominal columns.
 
    Args:
        col_a: First categorical column.
        col_b: Second categorical column.
 
    Returns:
        Tuple of (chi2_statistic, p_value).
    """
    table = pd.crosstab(col_a, col_b).values
    chi2, p, _, _ = chi2_contingency(table)
    return float(chi2), float(p)

In [6]:
def kappa_both_mappings(
    col_a: pd.Series,
    col_b: pd.Series,
    label_map_a: dict,
    label_map_b: dict,
) -> tuple[float, float]:
    """
    Compute Cohen's kappa under both possible label orientations.

    Mapping A uses label_map_a as given.
    Mapping B flips the 0/1 assignment for col_a only, which is equivalent
    to testing the opposite directional assumption.

    Args:
        col_a: First categorical column.
        col_b: Second categorical column.
        label_map_a: Encoding dict for col_a (e.g. {'@': 0, '@@': 1}).
        label_map_b: Encoding dict for col_b (e.g. {'R': 0, 'S': 1}).

    Returns:
        Tuple of (kappa_mapping_a, kappa_mapping_b).
    """
    # print("Cohen's kappa (agreement beyond chance)")
    # print("Interpretation: 1=perfect, 0=chance, -1=perfectly wrong")
    encoded_b = col_b.map(label_map_b).values

    # Mapping A: as given
    encoded_a_fwd = col_a.map(label_map_a).values
    kappa_fwd = cohen_kappa_score(encoded_a_fwd, encoded_b)

    # Mapping B: flip col_a encoding (equivalent to negating the direction)
    flipped_map_a = {k: 1 - v for k, v in label_map_a.items()}
    encoded_a_flipped = col_a.map(flipped_map_a).values
    kappa_flipped = cohen_kappa_score(encoded_a_flipped, encoded_b)

    # interp = (
    #     "strong agreement" if kappa_fwd > 0.6
    #     else "moderate agreement" if kappa_fwd > 0.3
    #     else "near-independent" if abs(kappa_fwd) < 0.1
    #     else "some agreement" if kappa_fwd > 0
    #     else "anti-correlated"
    # )
    # print(f"  {col_a} vs {col_b}: κ = {kappa:+.4f}  ({interp})")

    return float(kappa_fwd), float(kappa_flipped)

In [7]:
def mcc_and_pearson_both_mappings(
    col_a: pd.Series,
    col_b: pd.Series,
    label_map_a: dict,
    label_map_b: dict,
) -> tuple[float, float, float, float]:
    """
    Compute MCC and Pearson r under both label orientations for col_a.

    Args:
        col_a: First categorical column.
        col_b: Second categorical column.
        label_map_a: Encoding dict for col_a.
        label_map_b: Encoding dict for col_b.

    Returns:
        Tuple of (mcc_fwd, mcc_flipped, pearson_fwd, pearson_flipped).
    """
    encoded_b = col_b.map(label_map_b).values
    encoded_a_fwd = col_a.map(label_map_a).values # Mapping A: as given

    flipped_map_a = {k: 1 - v for k, v in label_map_a.items()}  # Mapping B: flip col_a encoding (equivalent to negating the direction)
    encoded_a_flipped = col_a.map(flipped_map_a).values

    mcc_fwd     = float(matthews_corrcoef(encoded_a_fwd,     encoded_b))
    mcc_flipped = float(matthews_corrcoef(encoded_a_flipped, encoded_b))
    r_fwd,   _  = pearsonr(encoded_a_fwd,     encoded_b)
    r_flipped, _= pearsonr(encoded_a_flipped, encoded_b)

    return mcc_fwd, mcc_flipped, float(r_fwd), float(r_flipped)

In [8]:
# scratch work. alternative way to calculate pearson r
# Pearson correlation on binary-encoded labels
TARGET_COLS = ["@/@@_class", "R/S_class", "F/L_class"]

# Binary encoding for correlation
label_maps = {
    "@/@@_class": {"@": 0, "@@": 1},
    "R/S_class":  {"R": 0, "S": 1},
    "F/L_class":  {"F": 0, "L": 1},
}
df_encoded = pd.DataFrame({
    col: df[col].map(label_maps[col]) for col in TARGET_COLS
})

print("\n" + "=" * 60)
print("Pearson correlation (binary-encoded labels)")
print("=" * 60)
print(df_encoded.corr().round(4).to_string())


Pearson correlation (binary-encoded labels)
            @/@@_class  R/S_class  F/L_class
@/@@_class      1.0000     0.0192     0.0555
R/S_class       0.0192     1.0000     0.0534
F/L_class       0.0555     0.0534     1.0000


## Label encodings

Default mapping: first-listed category → 0, second → 1.

For signed metrics (kappa, MCC, Pearson), we report under both orientations of `col_a` as a sensitivity check. A robust near-zero result under both mappings is strong evidence of independence.

In [9]:
TARGET_COLS = ["@/@@_class", "R/S_class", "F/L_class"]
LABEL_MAPS = {
    "@/@@_class": {"@": 0, "@@": 1},
    "R/S_class":  {"R": 0, "S":  1},
    "F/L_class":  {"F": 0, "L":  1},
}

PAIRS = [
    ("@/@@_class", "R/S_class"),
    ("@/@@_class", "F/L_class"),
    ("R/S_class",  "F/L_class"),
]

In [10]:
records = []

for col_a, col_b in PAIRS:
    # 1. Chi-square
    chi2_stat, chi2_p = chi_square(df[col_a], df[col_b])

    # 2. Cramér's V
    v, _              = cramers_v(df[col_a], df[col_b])

    # 3. Cohen's kappa — both mappings
    k_fwd, k_flipped = kappa_both_mappings(
        df[col_a], df[col_b], LABEL_MAPS[col_a], LABEL_MAPS[col_b]
    )

    # 4. and 5. MCC and Pearson — both mappings
    mcc_fwd, mcc_flip, r_fwd, r_flip = mcc_and_pearson_both_mappings(
        df[col_a], df[col_b], LABEL_MAPS[col_a], LABEL_MAPS[col_b]
    )

    records.append({
        "pair":             f"{col_a} vs {col_b}",
        "chi2":             round(chi2_stat, 4),
        "chi2_p":           round(chi2_p, 6),
        "cramers_v":        round(v, 4),
        "kappa_mapping_a":  round(k_fwd, 4),
        "kappa_mapping_b":  round(k_flipped, 4),
        "mcc_mapping_a":    round(mcc_fwd, 4),
        "mcc_mapping_b":    round(mcc_flip, 4),
        "pearson_mapping_a":round(r_fwd, 4),
        "pearson_mapping_b":round(r_flip, 4),
    })

results_df = pd.DataFrame(records).set_index("pair")
results_df

,chi2,chi2_p,cramers_v,kappa_mapping_a,kappa_mapping_b,mcc_mapping_a,mcc_mapping_b,pearson_mapping_a,pearson_mapping_b
pair,,,,,,,,,
@/@@_class vs R/S_class,1.3437,0.246382,0.0187,0.0192,-0.0192,0.0192,-0.0192,0.0192,-0.0192
@/@@_class vs F/L_class,11.6496,0.000642,0.0550,0.0555,-0.0555,0.0555,-0.0555,0.0555,-0.0555
R/S_class vs F/L_class,10.7869,0.001022,0.0529,0.0534,-0.0534,0.0534,-0.0534,0.0534,-0.0534


## Interpreting the results

### First look at Cramér's V

Cramér's V is the primary metric because it requires no label-mapping assumption.
It answers: *"Is there any association at all, and how strong is it?"*

Cohen (1988) benchmarks for 2×2 tables:

| V | Strength |
|---|---|
| ~0.10 | Small |
| ~0.30 | Medium |
| ~0.50 | Large |

### Then look at chi-square p-value

The p-value tells you whether the association is statistically distinguishable
from noise. **But with n=3858, almost any non-zero V will be significant.**
A significant p-value with V < 0.10 means "real but negligible" i.e., not important enough.

### Lastly, look at kappa / MCC (both mappings)

These add directionality. If both mappings produce near-zero values, the variables
are independent regardless of how you orient the labels. If one mapping gives
a large positive value and the other a large negative value, there is a real
directional association — you just need to decide which orientation is chemically meaningful.

In [11]:
# Confirming MCC = kappa = Pearson for balanced data
print("Confirming MCC = kappa = Pearson for perfectly balanced (50/50) binary variables:")
for col in TARGET_COLS:
    balance = df[col].value_counts(normalize=True)
    print(f"  {col}: {balance.to_dict()}")

print()
for row in records:
    diff = abs(row["mcc_mapping_a"] - row["kappa_mapping_a"])
    print(f"  {row['pair']}: |MCC - kappa| = {diff:.8f}  "
          f"{'✓' if diff < 1e-9 else '⚠'}")

Confirming MCC = kappa = Pearson for perfectly balanced (50/50) binary variables:
  @/@@_class: {'@': 0.5, '@@': 0.5}
  R/S_class: {'S': 0.5, 'R': 0.5}
  F/L_class: {'F': 0.5, 'L': 0.5}

  @/@@_class vs R/S_class: |MCC - kappa| = 0.00000000  ✓
  @/@@_class vs F/L_class: |MCC - kappa| = 0.00000000  ✓
  R/S_class vs F/L_class: |MCC - kappa| = 0.00000000  ✓


## Final interpretation

| Pair | chi2 p-value | Cramér's V | Interpretation |
|---|---|---|---|
| `@/@@` vs `R/S` | 0.246 (n.s.) | 0.019 | **Completely independent**. also κ ≈ 0|
| `@/@@` vs `F/L` | 0.0006 | 0.055 | Significant but **negligible effect size**. also κ < 0.1 |
| `R/S` vs `F/L` | 0.001 | 0.053 | Significant but **negligible effect size** also κ < 0.1 |

**All three columns are measuring essentially independent properties of the same molecules.**

This is the key scientific result that motivates the experimental ladder:

1. **`R/S` prediction** — R/S is the CIP absolute chemical chirality configuration.
    Tests whether the model understands atomic priority rules since this is directly hashed into chiral Morgan fingerprints.
2. **`@/@@` prediction** — reads a SMILES formatting token 
   This is a SMILES parity tag that depends on the order in which atoms are written in the SMILES string. It is a formatting convention, not a chemical property.
3. **`F/L` prediction** — predicts the 3D chromatographic consequence of chirality.
   The scientifically meaningful target. Tests whether the model understands what chirality *means* in 3D space.

Consequence for model evaluation:
  - Predicting R/S with includeChirality=True Morgan FP should be the easiest task. It requires implicit CIP priority reasoning from topology.
  - Predicting @/@@ with : the @/@@ token is directly hashed into the fingerprint at radius 0.
  - Predicting F/L requires understanding the 3D consequence of chirality.


Note on the reference paper: the paper states Morgan FPs encode CIP R/S:
```
While chirality is included in SMILES with the “@”/“@@” label as a parity tag internal to the SMILES string, the Morgan fingerprints specify the stereochemistry of chiral centers with the CIP R/S labels. First, we investigated the ability of the LSV descriptors to predict the CIP label with the aim of testing their chemical significance in relation to an implicit property that can be derived from the molecular structure. We then challenged the new descriptors with the QSPR task of predicting the elution order of enantiomers observed on the Chiralpak AD-H column (a classification task with two classes, first eluted or last eluted enantiomer of the pair), 
```
This is correct and is why predicting R/S is an easier target than @/@@.
The RDKit Morgan fingerprint encodes the absolute Cahn-Ingold-Prelog (CIP) (R)/(S) or (E)/(Z) configurations, not the raw relative @/@@ parity tags found in the SMILES string.


In [ ]:
# optionally save the results
# OUTPUT_PATH = "chirality_association_results.csv"
# results_df.to_csv(OUTPUT_PATH)
# print(f"Saved to {OUTPUT_PATH}")
# results_df